# 개별종목 조합B — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합B 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합B의 피처 값만 지정합니다.
import json

COMBINATION = 'B'
FEATURE_COLUMNS = (
    'ret_5',
    'ret_20',
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'dist_high_20',
    'dist_high_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합B 피처: ('ret_5', 'ret_20', 'sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'dist_high_20', 'dist_high_60')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4797,0.5012,-0.0216,0.3516,0.1801,0.2862
1,2,balanced,980,20150123,20150421,0.3810,0.3978,-0.0169,0.3580,0.2875,0.3372
2,3,balanced,1210,20151228,20160328,0.3635,0.3762,-0.0127,0.3541,0.2704,0.3235
3,4,balanced,1439,20161202,20170228,0.4529,0.4617,-0.0088,0.3930,0.2346,0.3328
4,5,balanced,1669,20171113,20180207,0.4007,0.3901,0.0107,0.3770,0.2679,0.3378
5,6,balanced,1899,20181024,20190118,0.4312,0.3725,0.0587,0.4237,0.3198,0.3843
6,7,balanced,2129,20190930,20191224,0.4307,0.4781,-0.0475,0.3454,0.2471,0.3238
7,8,balanced,2359,20200902,20201130,0.4105,0.3476,0.0629,0.4088,0.4084,0.4093
8,9,balanced,2589,20210806,20211105,0.3673,0.3916,-0.0243,0.3447,0.1689,0.2598
9,10,balanced,2818,20220714,20221012,0.3875,0.3454,0.0421,0.3798,0.2200,0.3074


,OOS 폴드 평균
accuracy,0.4061
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0092
macro_f1,0.3741
down_recall,0.2650
core_harmonic_mean,0.3324


재실행 명령: python scripts/run_stock_model_experiment.py
